# ***IMPORTAR LIBRERIAS***

In [1]:
# modelo: Codigo APC2 subidor(50261), Codigo centro(44), Codigo APC2, Fecha, Codigo Turno
# -> 50261,44,46000,20/01/2023,T05

#importar librerias para operar con excel, csv, os, exp. regulares y tiempo
import csv
import openpyxl
import re
import os
from datetime import datetime
#Para iterar por la lista y generar el csv
#from pandas import DataFrame as df

#importar librerias para GUI
from tkinter import ttk, messagebox, filedialog
import tkinter as tk
from functools import partial
from base64 import b64decode


#Incluir la localizacion de los ficheros con los datos de los turnos:
directory = os.getcwd()

# ***FUNCIONES***

In [2]:
def leer_csv_turnos(directory):
    #Definimos separador y archivo
    separator = ","
    path = directory + r"\turnos_GXO.csv"

    #Abrimos el archivo y leemos el contenido
    with open(path, encoding="utf-8") as file:
        next(file) #omitir encabezado
        turnos = []
        
        #lo dividimos
        for turno in file:
            turno = turno.rstrip("\n")
            #print(turno)
            column = turno.split(separator)
            #print(column)
            
            code_gxo = column[0]
            description = column[1]
            turn = column[2]
            client = column[3]
            
            #lo metemos en un diccionario        
            turnos.append({
            "codigo" : code_gxo,
            "descripcion" : description,
            "turno" : turn,
            "cliente" : client,
                        
            })
    return turnos

def leer_csv_censo(directory):
    #Definimos separador y archivo
    separator = ";"
    path = directory + r"\censo.csv"

    #Abrimos el archivo y leemos el contenido
    with open(path, encoding="utf-8") as file:
        next(file) #omitir encabezado
        censo = []
        
        #lo dividimos
        for person in file:
            person = person.rstrip("\n")
            column = person.split(separator)
            
            apc2_id = column[0]
            name = column[1]
            contract = column[2]
            client = column[3]
                        
            #lo metemos en un diccionario        
            censo.append({
            "codigo_APC2" : apc2_id,
            "nombre" : name,
            "contrato" : contract,
            "cliente" : client,
                        
            })
    return censo

def convert_date(input_fecha):
    # Expresión regular para validar el formato "aaaa-mm-dd"
    regex = r'^(\d{4})-(0[1-9]|1[0-2])-(0[1-9]|[1-2][0-9]|3[0-1])$'

    # Verificar si la fecha cumple con el formato esperado
    if re.match(regex, input_fecha):
    # Extraer los componentes de la fecha
        ano, mes, dia = re.match(regex, input_fecha).groups()
    #Devolver la fecha en el formato "dd/mm/aaaa"
        return f'{dia}/{mes}/{ano}'
    else:
        pass
    
def read_dates(init_col, finish_col, date_row, sheet):
    data = []

    for i in range(init_col, finish_col + 1): 
        cell_obj = sheet.cell(row = date_row, column = i)
        
        ean = str(cell_obj.value)
        ean =ean.split()
        date_finish =ean[0]
        #print(date_finish)
        d =convert_date(date_finish)
        #print(d)
        
        
        if d not in data:
            data.append(d)
        else:
            pass
        # result=[]
        # result = pd.unique(data)
    
    return data

def obtain_day_week(date):
    # Convierte la cadena de fecha a un objeto datetime
    date_obj = datetime.strptime(date, '%Y-%m-%d')
    
    # Obtiene el nombre del día de la semana en español
    days_week = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]
    day_name = days_week[date_obj.weekday()]

    return day_name

def turn_selector(day_name, code_turn, operative_name):
    turnos_permitidos = ["LVM", "LVT", "DJM", "DJT", "LVN", "DJN"]

    if code_turn not in turnos_permitidos:
        return None
    
    regex = r'([a-zA-Z])$'
    result = re.search(regex, code_turn)
    
    if result:
        ult_lettter = result.group(1)
        
        if validador_dias(code_turn, day_name):
            if ult_lettter == "M" and operative_name != "NIKE":
                return "M2"
            elif ult_lettter == "T" and operative_name != "NIKE":
                return "T3"
            elif ult_lettter == "M" and operative_name == "NIKE":
                return "MNN"
            elif ult_lettter == "T" and operative_name == "NIKE":
                return "T05"
            

def validador_dias(cturn, day):
    dias_semana = ["lunes", "martes", "miércoles", "jueves", "viernes"]
    dias_semana_DJM = ["domingo", "lunes", "martes", "miércoles", "jueves"]

    # Crear el diccionario asignando las claves "LVM" y "DJM" a los días correspondientes
    diccionario_turnos = {"LVM": dias_semana, "LVT": dias_semana, "LVN": dias_semana, "DJM": dias_semana_DJM, "DJT": dias_semana_DJM, "DJT": dias_semana_DJM}
    
    if cturn not in diccionario_turnos:
        return False  # La clave del turno no es válida

    return day.lower() in diccionario_turnos[cturn]

def obtener_nombres_hojas(archivo_excel):
    try:
        # Cargar el libro de Excel
        libro = openpyxl.load_workbook(archivo_excel)
        # Obtener los nombres de las hojas
        nombres_hojas = libro.sheetnames
        return nombres_hojas

    except Exception as e:
        #print(f"Error al leer el archivo Excel: {e}")
        return None

def agregar_a_csv(output_lines, csv_filename):
    # Comprobar si el archivo CSV existe
    if not os.path.exists(csv_filename):
        # Si no existe, crear el archivo CSV y escribir el encabezado si es necesario
        with open(csv_filename, 'w', newline='') as file:
            writer = csv.writer(file)
            # Escribir el encabezado si es necesario
            # writer.writerow(["Columna1", "Columna2", ...])

    # Abrir el archivo CSV en modo de escritura y añadir las líneas de salida
    with open(csv_filename, 'a', newline='') as file:
        writer = csv.writer(file)
        for line in output_lines:
            # Supongamos que las líneas de salida son listas o tuplas
            writer.writerow(line)



# ***FUNCIONES GUI***

In [3]:
def ret_selection(combo):
    # Obtener la opción seleccionada.
    selection = combo.get()
    if selection == "Elena : 50838":
        #print("1")
        label = ttk.Label(
    
            text="Elena : 50838",
            foreground="black",
            font=("Courier", 10),
        )
        label.place(x=25, y=250)
    
        return "50838"
        
    if selection == "Javi: 50261":
        #print("2")
        label = ttk.Label(
    
            text="Javi : 50261",
            foreground="black",
            font=("Courier", 10),
        )
        label.place(x=25, y=250)
        return "50261"
        
    if selection == "Otro":
        messagebox.showinfo(
        message="Opción en Desarrollo",
        title="Selección", 
        )
    

    
def person_selector_to_up():
    app2 = tk.Toplevel()
    app2.geometry("400x200")
    app2.title("Selector de Persona")
    
    label = ttk.Label(
    app2,
    text="Seleccione Usuario que figurará como subidor",
    #textvariable=palabra,
    foreground="black",
    #bg="#E5E7E9",
    #justify="center",
    )
    
    label.place(x=75, y=20)

    combo = ttk.Combobox(
        app2,
        state="readonly",
        values=["Elena : 50838", "Javi: 50261", "Otro"],
        justify="center"
        
        )
    combo.place(x=125, y=50)
    button = ttk.Button(
        app2,
        text="Guardar",
        command=partial(ret_selection, combo),
        
        
        )
    button.place(x=100, y=100)   

    button2 = ttk.Button(
        app2,
        text="Salir",
        command=app2.destroy 
        
        )
    button2.place(x=200, y=100) 
    
    
    app2.focus()
    app2.grab_set()  
    
def in_progress():
    messagebox.showinfo(
        message="Opción en Desarrollo",
        title="In progress", 
        )
    
    
def abrir_fichero():
    fichero = filedialog.askopenfilename(
        title="Abrir un fichero Operaciones",
        initialdir="C:", 
        filetypes=(
        ("Ficheros de Excel", "*.xlsx"),
        ("Todos los ficheros","*.*")
    ), 
        
            
        )
    leer_excel(fichero)
    
    #return fichero
        
    # nombre_archivo, extension = os.path.splitext(fichero)
    # if extension == ".xlsx" or extension == ".xls":
    #     print(fichero)
    #     return fichero
    # else:
    #     messagebox.showerror(
    #         message="No es un archivo Excel",
    #         title="Error",
    #     )
    
def center_selector_to_up():
    app3 = tk.Toplevel()
    app3.geometry("400x200")
    app3.title("Selector de Centro")
    
    label = ttk.Label(
    app3,
    text="Seleccione Centro",
    foreground="black",
       
    )
    
    label.place(x=75, y=20)

    combo = ttk.Combobox(
        app3,
        state="readonly",
        values=["Cabanillas : 44", "Otro"],
        justify="center"
        
        )
    combo.place(x=125, y=50)
    button = ttk.Button(
        app3,
        text="Guardar",
        command=partial(ret_center, combo),
        
        
        )
    button.place(x=100, y=100)   

    button2 = ttk.Button(
        app3,
        text="Salir",
        command=app3.destroy,
        
        )
    button2.place(x=200, y=100) 
    
    
    app3.focus()
    app3.grab_set() 
    
def ret_center(combo):
    # Obtener la opción seleccionada.
    selection = combo.get()
    if selection == "Cabanillas : 44":
        #print("1")
        label = ttk.Label(
    
            text="Cabanillas : 44",
            foreground="black",
            font=("Courier", 10),
        )
        label.place(x=25, y=270)
    
        return "44"
        
        
    if selection == "Otro":
        messagebox.showinfo(
        message="Opción en Desarrollo",
        title="Selección", 
        )     
    

# ***LEER EL EXCEL***

In [4]:

def leer_excel(path):
    #print(directory)
    #path = directory + r"\Plantilla_Turnos_OPS.xlsx"
    #print(str(path))

    # Se abre el workbook y la hoja seleccionados (donde se guardo el documento por ultima vez)
    wb_obj = openpyxl.load_workbook(path, data_only=True)
    
    
    sheet = wb_obj.sheetnames
    sheet = wb_obj["ENERO"]

    # Obtener el numero de filas y columnas usadas
    row_limit = sheet.max_row
    column_limit = sheet.max_column

    print("Total de filas:", row_limit)
    print("Total de columnas:", column_limit)

    operative_name = sheet.cell(row = 2, column= 3).value

    print(operative_name)

    #Introducir el numero de columna en la que se encuentra el nombre (A=1, B=2, C=3...)
    name_col = 2

    #Introducir el numero de la fila en la que se encuentra el primer valor del nombre
    first_name_row = 7

    #Introducir el numero de la fila en la que se encuentra el primer valor del nombre
    apc2_col =1

    #Introducir el numero de columna en la que se encuentra la fecha
    date_col= 3

    #Introducir el numero de fila en la que se encuentra la primera fecha
    date_row = 5

    
    return operative_name, name_col, first_name_row, apc2_col, date_col, date_row, sheet

def leer_excel_censo(path, apc2code):
    # Se abre el workbook y la hoja seleccionados (donde se guardo el documento por ultima vez)
    wb_obj = openpyxl.load_workbook(path, data_only=True)
    sheet = wb_obj.sheetnames
    sheet = wb_obj["CENSO"]
    
    # Obtener el numero de filas y columnas usadas
    row_limit = sheet.max_row
    column_limit = sheet.max_column

    print("Total de filas:", row_limit)
    print("Total de columnas:", column_limit)

    #Introducir el numero de columna en la que se encuentra el numero de APC2 (A=1, B=2, C=3...)
    apc2_col = 1

    #Introducir el numero de la fila en la que se encuentra el tipo de contrato:
    contract_col =3
    
    # primera linea de APC2 code
    first_apc2_row=2
    
    for i in range(first_apc2_row, row_limit + 1): 
        cell_obj = sheet.cell(row = i, column = apc2_col)
        valid = str(cell_obj.value)
        
        if valid == apc2code:
            valid_obj = sheet.cell(row=i, column = contract_col)
            val_result = str(valid_obj.value)
            return val_result
        else:
            pass
            
        



# ***WORKBENCH***

In [5]:


#valor_encontrado = turn_selector("jueves", "DJM", operative_name)

#print(valor_encontrado)

# lines_to_add = [
#     ("Dato1", "Dato2", "Dato3"),
#     ("Dato4", "Dato5", "Dato6"),
#     # ...
# ]

# csv_file = "mi_archivo.csv"
# agregar_a_csv(lines_to_add, csv_file)

file = "\Censo_2023.xlsx"
path = directory + file
print(path)


leer_excel_censo(path, "28207")





c:\Users\jgmeras\Documents\Python Scripts\6. CABANILLAS\Censo_2023.xlsx
Total de filas: 177
Total de columnas: 4


'DIRECT'

# ***GUI***

In [6]:
app = tk.Tk()

style = ttk.Style()
raw_image = "iVBORw0KGgoAAAANSUhEUgAAAHwAAAAuCAMAAADdho1wAAAAV1BMVEVHcEz/OgD/OgD/OgD/OgD/OgD/OgD/OgD/OgD/OgD/OgD/OgD/OgD/OgD/OgD/OgD/OgD/OgD/OgD/OgD/OgD/OgD/OgD/OgD/OgD/OgD/OgD/OgD/OgD/maxbAAAAHHRSTlMAOZ4DJi3nBvftGwy7zNVo3RNcRk20w4eoe5JvWvIB4QAAA/5JREFUWMPNWNuypCoMbS94Q1QUUdT//85x79Y2IRG7ps6pmlQ/AWZ1FrmR1+tfk0RqrWX+H2nL5dfKdNQuo1VK2TF2pkqunUpckiJleQq2JNa2vrXVsRuqB5OjSZXbJUU3959Phq78SNbDr/rs2rEV0lYAZU08yHvsKIbIhzTuUJfHYFUBM9IGIPQhbUVtbvjXEwP9I6OkINt0GQj/VHwq1y7jdBVxyppteeitWM8jPSCxjM5VA1abU7UYb7RtzUCxTXd3ev5cFCL+5KNSDOlRs91K2RPs7O5sJ25ut32vTQzpUbdt36Pfnz4xKPHdL8VRSUlPmy0oJWK+stsz6YT4ZU8CcqSky3F7kAZ4XbL45pZZVhDSf06mgKLCvF4rQ7rzte3iLcVXxA0oxgrlBpGKqI87GLaHtECP1aKj5ogMh7YzUWRcjfB//vfB5YwcrNVvI/fbcBNJSRIedjHj6YhGaw4N0lgudyCf2SziOaExCY2FdJ5MIsNjkAgr+LeLgfmrSjyWnrYI+hC88VmjFArR42OtuQsCXhDxhHRpeaf2Q7B5czJA7pIvqr1gksLHfQW4w9X/dCW8r1zCDsoaCNweuC4p4BUw3f2uAI+10LmHtb1kNYATkkVARE7kWqEs3m5ec3VyjwGc7ev8fg/mDGBKyzirp1Aq9rhvHQJ/rcVNtsxHGk3wX1/fKek5O8hnfo7E4FrBvQUk/4vHgnEg4I6N9sDNLbEeuEEJuRFfgic+OEu7rrcQeKW80pdztJtn2lmHc1sInFbB677mQJijQHwrBP455lyZI+CGJNguZUJtTh5DDVqZvS8vd/X4lowD90lHwQZsy9JQklmJHe64uVNiBpyQjoiH6XUKhPkRiFXHd4v4Si5wUwaqWgILSxYFCoumTf+on8BRq/zQRNnqthwunP/E+gEckF442rPDUN6/AtdeQSM/JUyiBqeOLicF4X6CQ9LnvKevFUTk1hxNmd9GXZmhR6FTxqaS+f6qFqsioQZJ36+UtNIkORZq6gfSQILanXu9SdHZcaybksZ5At8nzu/oDH3DnL2ztwLjQDRPbf4BDkm32m8sjtaI5mZfsCOa8itwSPrR7iGkg3jRhXV1OAST/gn9p7tDpB9AKBWfxA9ZSFVGCn0fPF/8NuBDyTUQEzOwGAK2d0yTMajA+T73npNXHUtQqj/5uB017KHM9aTVckN9tqTE0+HjFQ0sziJe8UOWctJ3s6iZ+aBbBL3cDNYAFKmfSRGnrZwDvXkeTdYbhbXp8WLT4/5uPgX3CaIBWw5qQ6MwNUUPs0Ap+mmurVJ2XtYBDAFlCsR7vFZgC84NPyNFOy5tpL94De2/XO6SvP5CEm6Y+rfK/lf5A5RpN2/mmB9oAAAAAElFTkSuQmCC"

#definimos dimensiones : ancho x alto
#app.config(width=400, height=300)
app.geometry("450x300")

app.title("Generador Archivos CSV de Turnos Masivos")

#cambiamos tirulo de ventana

# progressbar = ttk.Progressbar(orient=tk.VERTICAL)
# progressbar.place(x=15, y=60, height=160)

label = ttk.Label(
    app,
    text="Comenzar con la selección",
    foreground="black",
    font=("Courier", 14),
    #bg="#E5E7E9",
    #justify="center",
    
    
)
label.place(x=70, y=50)

button = ttk.Button(
    app,
    text="Selector de persona para subir",
    command=person_selector_to_up,
    
)
button.place(x=50, y=100)

button2 = ttk.Button(
    app,
    text="Seleccionar Centro",
    command=center_selector_to_up,
)
button2.place(x=250, y=100)

button3 = ttk.Button(
    app,
    text="Cargar datos de Excel",
    command=abrir_fichero,
)
button3.place(x=50, y=150)

button4 = ttk.Button(
    app,
    text="Generar CSV",
    command=in_progress,
)
button4.place(x=250, y=150)

button5 = ttk.Button(
    app,
    text="Salir",
    command=app.destroy

)
button5.place(x=175, y= 200)

image = tk.PhotoImage(data=b64decode(raw_image))
label2 = ttk.Label(
    image=image
    )
label2.place(x=275, y=225)




app.mainloop()